# Tasks

4.1 predict magnitude of antibody response - H1N1 A/Victoria/4897/2022 (D28)

4.2 predict magnitude of antibody response - H3N2 A/Massachusetts/18/2022 (D28)

4.3 predict magnitude of antibody response - Vic B/Austria/1359417/2021 (D28)

4.4 predict magnitude of antibody response - all 3 vaccine strains (D28)

**4.5 predict antibody breadth - all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI
* Measure: Geo mean
* Metric: Spearman correlation
* Full description: Geomean HAI across all variants

4.6 predict antibody breadth - all variants (D28)

4.7 predict antibody durability - H1N1 A/Victoria/4897/2022 (D365)

4.8 predict antibody durability - H3N2 A/Massachusetts/18/2022 (D365)

4.9 predict antibody durability - Vic B/Austria/1359417/2021 (D365)

4.10 predict antibody durability - all 3 vaccine strains (D365)

## Task 4.5: Focus
Breadth refers to how widely an antibody response covers different variants of a pathogen (not just the specific strain, but also related versions).
Want to measure a person's antibody response broadly, so we are predicting the 'average protective coverage' across the whole panel of flu variants.
Use the geometric mean (average used for antibody titers)

In [19]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

In [20]:
DATA_PATH = 'data/PART2-26-01-26_reorg/PART2-26-01-26_reorg'
train_hai = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
train_participants = pd.read_csv(DATA_PATH + '/train_participants.tsv', sep='\t')

tables = {
    'train_hai': train_hai,
    'train_participants': train_participants,
}

In [21]:
for name, df in tables.items():
    print(f"\n{'=' * 50}")
    print(f'TABLE: {name}')
    print(f"{'=' * 50}")
    display(df.head(5))


TABLE: train_hai


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown



TABLE: train_participants


,participant_id,subject,biological_sex,race,min_age,max_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description,basic_curation,pubmed_ids,main_pmid,main_publication_author
0,SDY269.SUB112836,SUB112836,female,White,28,28,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
1,SDY269.SUB112849,SUB112849,female,Black or African American,39,39,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
2,SDY269.SUB112854,SUB112854,male,Black or African American,46,46,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
3,SDY269.SUB112860,SUB112860,female,White,32,32,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
4,SDY269.SUB112881,SUB112881,female,Black or African American,29,29,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)


### Data Cleaning
HAI
* Timepoint column only missing 1.89% of values, and value column missing 0.02% of values. We decide to drop these rows.
* We also make sure the value column is numeric.

Participants
* main_publication_author missing 47.55% of values, geolocation missing 20.18%, pubmed_ids missing 3.06%, main_pmid missing 3.06%
* We drop the publication column all together. Will address other missing values depending on task.

In [22]:
# For HAI we can dropna as the number of missing values is low. Want to make sure the value is numeric.
train_hai.dropna(inplace=True)
train_hai['value'] = pd.to_numeric(train_hai['value'], errors='coerce')
# Drop weird value
train_hai = train_hai[train_hai['virus_strain'] != '-']

# For Participants drop publication column all together
train_participants.drop(columns=['main_publication_author'], inplace=True)
# Fix capital and lower case issues
train_participants['biological_sex'] = train_participants['biological_sex'].str.lower().str.strip()
# Consistent names for race
train_participants['race'] = train_participants['race'].replace('race: unknown', 'Unknown')
train_participants['race'] = train_participants['race'].replace('race: other', 'Unknown')

In [6]:
hai_cols_to_keep = [
    'hai_id', 'participant_id', 'timepoint', 'virus_strain', 'value'
]
participant_cols_to_keep = [
    'participant_id', 'biological_sex', 'race', 'min_age', 'geolocation', 'investigation_id', 'investigation_name',
    'arm_id', 'arm_name', 'data_source', 'description'
]

train_merged = train_hai[hai_cols_to_keep].merge(
    train_participants[participant_cols_to_keep],
    on='participant_id',
    how='left'
)

train_merged.dropna(inplace=True)  # for simplicity
train_merged.head()

,hai_id,participant_id,timepoint,virus_strain,value,biological_sex,race,min_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.0,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.0,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.0,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.0,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.0,female,race: unknown,28,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone


### Build the target variable (y) - Geomean of HAI at Day 28

In [7]:
timepoint_counts = train_merged['timepoint'].value_counts()
print(timepoint_counts)

timepoint
 0.0      46365
 28.0     45401
 365.0    18141
 90.0      2190
 14.0       767
 30.0       468
 3.0        318
 75.0       318
 7.0        197
 70.0       180
 180.0       51
-7.0         18
Name: count, dtype: int64


In [8]:
d28 = train_merged[train_merged['timepoint'] == 28.0].copy()
y = (
    d28.groupby('participant_id')['value']
    .apply(lambda x: gmean(x))
    .reset_index()
    .rename(columns={'value': 'geomean_d28'})
)

In [9]:
y

,participant_id,geomean_d28
0,2016_UGA.ID_001,122.696422
1,2016_UGA.ID_002,42.430638
2,2016_UGA.ID_003,132.085872
3,2016_UGA.ID_004,33.021468
4,2016_UGA.ID_005,43.061034
...,...,...
2872,SDY67.SUB113607,320.000000
2873,SDY67.SUB113608,226.274170
2874,SDY67.SUB113609,113.137085
2875,SDY67.SUB113610,160.000000


### Build the features (X) - Day 0 of HAI and demographics

In [10]:
# Filter to Day 0 — the baseline readings before vaccination
d0 = train_merged[train_merged['timepoint'] == 0.0].copy()

# Pivot: one row per participant, one column per virus strain
# So instead of long format (many rows per person), we get a wide format (one row per person)
d0_wide = d0.pivot_table(
    index='participant_id',
    columns='virus_strain',
    values='value',
    aggfunc='mean'  # if duplicate entries exist, average them
).reset_index()

d0_wide.head()

virus_strain,participant_id,-,Anc B/Lee/1940,Anc B/Maryland/1959,Anc B/Singapore/1964,H1N1 A/Beijing/262/1995,H1N1 A/Brazil/11/1978,H1N1 A/Brisbane/2/2018,H1N1 A/Brisbane/59/2007,H1N1 A/California/7/2009,...,Vic B/Washington/2/2019,Yam B/Brisbane/3/2007,Yam B/Florida/4/2006,Yam B/Harbin/7/1994,Yam B/Massachusetts/2/2012,Yam B/Phuket/3073/2013,Yam B/Sichuan/379/1999,Yam B/Texas/6/2011,Yam B/Wisconsin/1/2010,Yam B/Yamagata/16/1988
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,40.0,5.0,NaN,10.0,160.0,...,NaN,NaN,640.0,640.0,640.0,320.0,640.0,160.0,320.0,160.0
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,80.0,5.0,NaN,40.0,20.0,...,NaN,NaN,80.0,160.0,80.0,80.0,160.0,40.0,80.0,40.0
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,80.0,5.0,NaN,10.0,160.0,...,NaN,NaN,640.0,1280.0,640.0,640.0,640.0,320.0,320.0,320.0
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,10.0,5.0,NaN,10.0,10.0,...,NaN,NaN,80.0,160.0,80.0,40.0,320.0,20.0,40.0,10.0
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,10.0,5.0,NaN,5.0,40.0,...,NaN,NaN,160.0,160.0,160.0,80.0,320.0,80.0,80.0,40.0


In [11]:
d0_wide.columns

Index(['participant_id', '-', 'Anc B/Lee/1940', 'Anc B/Maryland/1959',
       'Anc B/Singapore/1964', 'H1N1 A/Beijing/262/1995',
       'H1N1 A/Brazil/11/1978', 'H1N1 A/Brisbane/2/2018',
       'H1N1 A/Brisbane/59/2007', 'H1N1 A/California/7/2009',
       'H1N1 A/Chile/1/1983', 'H1N1 A/Denver/1/1957',
       'H1N1 A/Fort Monmouth/1/1947', 'H1N1 A/Guangdong-Maonan/SWL1536/2019',
       'H1N1 A/Hawaii/70/2019', 'H1N1 A/Michigan/45/2015',
       'H1N1 A/New Caledonia/20/1999', 'H1N1 A/New Jersey/8/1976',
       'H1N1 A/Singapore/6/1986', 'H1N1 A/Solomon Islands/3/2006',
       'H1N1 A/South Carolina/1/1918', 'H1N1 A/South Dakota/6/2007',
       'H1N1 A/Texas/36/1991', 'H1N1 A/USSR/90/1977',
       'H1N1 A/Victoria/2570/2019', 'H1N1 A/Victoria/4897/2022',
       'H1N1 A/Weiss/JY2/1943', 'H3N2 A/Brisbane/10/2007',
       'H3N2 A/Darwin/9/2021', 'H3N2 A/Hong Kong/1/1968',
       'H3N2 A/Hong Kong/2671/2019', 'H3N2 A/Hong Kong/4801/2014',
       'H3N2 A/Kansas/14/2017', 'H3N2 A/Mississippi/1/

In [12]:
# Log-transform the HAI columns (brings them to a linear scale)
hai_cols = [c for c in d0_wide.columns if c != 'participant_id']
d0_wide[hai_cols] = np.log1p(d0_wide[hai_cols])  # log1p = log(1+x), handles near-zero values
# Get demographics — one row per participant (drop duplicates since each person appears many times)
demo = train_merged[['participant_id', 'biological_sex', 'race', 'min_age']].drop_duplicates()
# Encode categorical variables — models need numbers, not text
demo = pd.get_dummies(demo, columns=['biological_sex', 'race'], drop_first=True)
# Merge Day 0 HAI + demographics into one feature table
X_df = d0_wide.merge(demo, on='participant_id', how='inner')
X_df.head()

,participant_id,-,Anc B/Lee/1940,Anc B/Maryland/1959,Anc B/Singapore/1964,H1N1 A/Beijing/262/1995,H1N1 A/Brazil/11/1978,H1N1 A/Brisbane/2/2018,H1N1 A/Brisbane/59/2007,H1N1 A/California/7/2009,...,Yam B/Wisconsin/1/2010,Yam B/Yamagata/16/1988,min_age,biological_sex_male,race_Asian,race_Black or African American,race_Multiracial,race_White,race_race: other,race_race: unknown
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,3.713572,1.791759,NaN,2.397895,5.081404,...,5.771441,5.081404,29,False,False,False,False,False,False,True
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,4.394449,1.791759,NaN,3.713572,3.044522,...,4.394449,3.713572,29,False,False,False,False,False,False,True
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,4.394449,1.791759,NaN,2.397895,5.081404,...,5.771441,5.771441,28,False,False,False,False,False,False,True
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,2.397895,1.791759,NaN,2.397895,2.397895,...,3.713572,2.397895,27,True,False,False,False,False,False,True
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,2.397895,1.791759,NaN,1.791759,3.713572,...,4.394449,3.713572,25,False,False,False,False,False,False,True
